# OR · 08 Production Scheduling


## 1️⃣ Configuración del Entorno

## 🎯 Objetivos de Aprendizaje

- Definir qué aprenderá el lector (máx. 5–7 puntos).
- Conectar con el caso de uso del dominio (demanda, logística, IoT).
- Incluir resultados verificables (métricas, validaciones, artefactos generados).

In [ ]:
import pandas as pd
import numpy as np
import pulp
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

# Rutas
DATA_DIR = Path("../../data/raw")
OUTPUT_DIR = Path("../../data/processed/or08_production_schedule")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("✅ Librerías cargadas")
print(f"   PuLP versión: {pulp.__version__}")
print(f"📁 Directorio datos: {DATA_DIR.resolve()}")

## 2️⃣ Generar Datos de Planificación

In [ ]:
# Parámetros del problema
np.random.seed(42)

products = ['SKU-A', 'SKU-B', 'SKU-C', 'SKU-D', 'SKU-E']
periods = list(range(1, 7))  # 6 meses

# Costos y tiempos por producto
df_products = pd.DataFrame({
    'product': products,
    'production_cost': [50, 60, 45, 70, 55],  # $/unidad
    'inventory_cost': [2, 3, 1.5, 4, 2.5],     # $/unidad/mes
    'machine_hours': [0.5, 0.8, 0.4, 1.0, 0.6] # horas/unidad
})

# Demanda por producto y periodo
np.random.seed(42)
demand_data = []
for product in products:
    base_demand = np.random.randint(100, 300)
    for period in periods:
        # Demanda con estacionalidad
        seasonal_factor = 1 + 0.3 * np.sin(2 * np.pi * period / 12)
        demand = int(base_demand * seasonal_factor * np.random.uniform(0.8, 1.2))
        demand_data.append({
            'product': product,
            'period': period,
            'demand': demand
        })

df_demand = pd.DataFrame(demand_data)

# Capacidades
MACHINE_CAPACITY = 1000  # horas/mes disponibles
STORAGE_CAPACITY = 2000  # unidades totales

print("📊 Datos de Planificación:")
print("\n📦 Productos:")
display(df_products)
print("\n📈 Demanda (primeros 10 registros):")
display(df_demand.head(10))

## 3️⃣ Visualizar Demanda

In [ ]:
# Pivot para visualización
df_demand_pivot = df_demand.pivot(index='period', columns='product', values='demand')

fig = go.Figure()
for product in products:
    fig.add_trace(go.Scatter(
        x=df_demand_pivot.index,
        y=df_demand_pivot[product],
        mode='lines+markers',
        name=product
    ))

fig.update_layout(
    title="Demanda por Producto y Periodo",
    xaxis_title="Periodo (Mes)",
    yaxis_title="Demanda (unidades)",
    hovermode='x unified'
)
fig.show()

print("📊 Demanda Total por Periodo:")
print(df_demand.groupby('period')['demand'].sum())

## 4️⃣ Formulación del Modelo LP

### Variables de Decisión:
- `X[p, t]`: Cantidad a producir del producto `p` en periodo `t`
- `I[p, t]`: Inventario del producto `p` al final de periodo `t`

### Función Objetivo:
```
Minimizar: Σ (production_cost[p] * X[p,t] + inventory_cost[p] * I[p,t])
```

### Restricciones:
1. **Balance de Inventario**: `I[p,t] = I[p,t-1] + X[p,t] - Demand[p,t]`
2. **Capacidad de Máquina**: `Σ (machine_hours[p] * X[p,t]) ≤ MACHINE_CAPACITY`
3. **Capacidad de Almacén**: `Σ I[p,t] ≤ STORAGE_CAPACITY`
4. **No Negatividad**: `X[p,t], I[p,t] ≥ 0`

In [ ]:
# Crear modelo
model = pulp.LpProblem("Production_Scheduling", pulp.LpMinimize)

# Variables de decisión
production = pulp.LpVariable.dicts(
    "Production",
    ((p, t) for p in products for t in periods),
    lowBound=0,
    cat='Continuous'
)

inventory = pulp.LpVariable.dicts(
    "Inventory",
    ((p, t) for p in products for t in periods),
    lowBound=0,
    cat='Continuous'
)

print("✅ Variables de decisión creadas")
print(f"   - Production: {len(products) * len(periods)} variables")
print(f"   - Inventory: {len(products) * len(periods)} variables")

## 5️⃣ Función Objetivo

In [ ]:
# Diccionarios para lookup rápido
prod_cost = df_products.set_index('product')['production_cost'].to_dict()
inv_cost = df_products.set_index('product')['inventory_cost'].to_dict()

# Función objetivo: minimizar costo total
model += (
    pulp.lpSum(
        prod_cost[p] * production[(p, t)] + inv_cost[p] * inventory[(p, t)]
        for p in products for t in periods
    ),
    "Total_Cost"
)

print("✅ Función objetivo definida: Minimizar (Costo Producción + Costo Inventario)")

## 6️⃣ Restricciones

In [ ]:
# Diccionario de demanda
demand_dict = df_demand.set_index(['product', 'period'])['demand'].to_dict()
machine_hours = df_products.set_index('product')['machine_hours'].to_dict()

# 1. Balance de inventario
for p in products:
    for t in periods:
        if t == 1:
            # Primer periodo: inventario inicial = 0
            model += (
                inventory[(p, t)] == production[(p, t)] - demand_dict[(p, t)],
                f"Balance_{p}_t{t}"
            )
        else:
            # Periodos siguientes
            model += (
                inventory[(p, t)] == inventory[(p, t-1)] + production[(p, t)] - demand_dict[(p, t)],
                f"Balance_{p}_t{t}"
            )

# 2. Capacidad de máquina
for t in periods:
    model += (
        pulp.lpSum(machine_hours[p] * production[(p, t)] for p in products) <= MACHINE_CAPACITY,
        f"Machine_Capacity_t{t}"
    )

# 3. Capacidad de almacén
for t in periods:
    model += (
        pulp.lpSum(inventory[(p, t)] for p in products) <= STORAGE_CAPACITY,
        f"Storage_Capacity_t{t}"
    )

print("✅ Restricciones agregadas:")
print(f"   - Balance de inventario: {len(products) * len(periods)}")
print(f"   - Capacidad de máquina: {len(periods)}")
print(f"   - Capacidad de almacén: {len(periods)}")
print(f"\n📊 Total de restricciones: {len(model.constraints)}")

## 7️⃣ Resolver Modelo

In [ ]:
# Resolver
print("⏳ Resolviendo modelo...")
solver = pulp.PULP_CBC_CMD(msg=0)  # Silenciar output del solver
model.solve(solver)

# Status
status = pulp.LpStatus[model.status]
print(f"\n✅ Status: {status}")

if status == 'Optimal':
    print(f"💰 Costo Total Óptimo: ${pulp.value(model.objective):,.2f}")
else:
    print("❌ No se encontró solución óptima")

## 8️⃣ Extraer Resultados

In [ ]:
# Extraer plan de producción
production_plan = []
for p in products:
    for t in periods:
        production_plan.append({
            'product': p,
            'period': t,
            'production': production[(p, t)].varValue,
            'inventory': inventory[(p, t)].varValue,
            'demand': demand_dict[(p, t)]
        })

df_plan = pd.DataFrame(production_plan)

print("📋 Plan de Producción (primeros 10 registros):")
display(df_plan.head(10))

# Guardar
df_plan.to_csv(OUTPUT_DIR / "production_plan.csv", index=False)
print(f"\n💾 Plan guardado: {OUTPUT_DIR / 'production_plan.csv'}")

## 9️⃣ Análisis de Resultados

In [ ]:
# Plan de producción por producto
df_prod_pivot = df_plan.pivot(index='period', columns='product', values='production')

fig = go.Figure()
for product in products:
    fig.add_trace(go.Bar(
        x=df_prod_pivot.index,
        y=df_prod_pivot[product],
        name=product
    ))

fig.update_layout(
    title="Plan de Producción Óptimo",
    xaxis_title="Periodo (Mes)",
    yaxis_title="Producción (unidades)",
    barmode='stack'
)
fig.show()

# Inventario al final de cada periodo
df_inv_pivot = df_plan.pivot(index='period', columns='product', values='inventory')

fig2 = go.Figure()
for product in products:
    fig2.add_trace(go.Scatter(
        x=df_inv_pivot.index,
        y=df_inv_pivot[product],
        mode='lines+markers',
        name=product,
        stackgroup='one'
    ))

fig2.update_layout(
    title="Inventario al Final de Cada Periodo",
    xaxis_title="Periodo (Mes)",
    yaxis_title="Inventario (unidades)"
)
fig2.show()

## 🔟 Validación de Restricciones

In [ ]:
# Verificar uso de capacidad de máquina
machine_usage = []
for t in periods:
    total_hours = sum(
        machine_hours[p] * production[(p, t)].varValue
        for p in products
    )
    machine_usage.append({
        'period': t,
        'hours_used': total_hours,
        'capacity': MACHINE_CAPACITY,
        'utilization': total_hours / MACHINE_CAPACITY * 100
    })

df_machine = pd.DataFrame(machine_usage)

fig = go.Figure()
fig.add_trace(go.Bar(
    x=df_machine['period'],
    y=df_machine['hours_used'],
    name='Horas Usadas',
    marker_color='lightblue'
))
fig.add_hline(
    y=MACHINE_CAPACITY,
    line_dash="dash",
    line_color="red",
    annotation_text="Capacidad Máxima"
)
fig.update_layout(
    title="Uso de Capacidad de Máquina",
    xaxis_title="Periodo",
    yaxis_title="Horas"
)
fig.show()

print("⚙️ Utilización de Máquina:")
display(df_machine)

# Verificar almacén
storage_usage = df_plan.groupby('period')['inventory'].sum().reset_index()
storage_usage['capacity'] = STORAGE_CAPACITY
storage_usage['utilization'] = storage_usage['inventory'] / STORAGE_CAPACITY * 100

print("\n📦 Utilización de Almacén:")
display(storage_usage)

## 1️⃣1️⃣ Desglose de Costos

In [ ]:
# Calcular costos por componente
df_plan['prod_cost'] = df_plan.apply(
    lambda row: prod_cost[row['product']] * row['production'], axis=1
)
df_plan['inv_cost'] = df_plan.apply(
    lambda row: inv_cost[row['product']] * row['inventory'], axis=1
)

total_prod_cost = df_plan['prod_cost'].sum()
total_inv_cost = df_plan['inv_cost'].sum()

# Pie chart de desglose
fig = go.Figure(data=[
    go.Pie(
        labels=['Costo Producción', 'Costo Inventario'],
        values=[total_prod_cost, total_inv_cost],
        hole=0.3
    )
])
fig.update_layout(title="Desglose de Costos Totales")
fig.show()

print("💰 Resumen de Costos:")
print(f"   - Producción: ${total_prod_cost:,.2f} ({total_prod_cost/(total_prod_cost+total_inv_cost)*100:.1f}%)")
print(f"   - Inventario: ${total_inv_cost:,.2f} ({total_inv_cost/(total_prod_cost+total_inv_cost)*100:.1f}%)")
print(f"   - TOTAL: ${total_prod_cost + total_inv_cost:,.2f}")

## 🎓 Conclusiones

**Aprendizajes Clave:**
1. ✅ **Programación Lineal**: PuLP permite modelar problemas complejos de forma declarativa
2. ✅ **Trade-offs**: Modelo balancea costos de producción vs inventario automáticamente
3. ✅ **Capacidades**: Restricciones aseguran factibilidad operativa
4. ✅ **Solver CBC**: Resuelve modelos con decenas de variables en segundos

**Resultados de Negocio:**
- 💰 Costo óptimo: ~${total_prod_cost + total_inv_cost:,.0f}
- ⚙️ Utilización de máquina: {df_machine['utilization'].mean():.1f}% promedio
- 📦 Inventario promedio: {df_plan['inventory'].mean():.0f} unidades
- 🎯 100% de cumplimiento de demanda

**Decisiones Operativas:**
- Producir en periodos de baja demanda para aprovechar capacidad
- Mantener inventario buffer solo si costo de inventario < costo de overtime
- Productos con alto `machine_hours` priorizarse en periodos de baja demanda agregada

**Próximos Pasos:**
- Agregar costos de setup entre productos
- Incluir variables binarias para decisiones de producción (MIP)
- Multi-planta con costos de transporte (ver OR-09)
- Incertidumbre con programación estocástica

---

**🔗 Notebooks Relacionados:**
- [OR-01: Stock de Seguridad](../50_optimization_or/OR-01-stock_seguridad.ipynb)
- [OR-02: Políticas de Inventario](../50_optimization_or/OR-02-politicas_inventario.ipynb)
- [OR-09: Red Logística](../50_optimization_or/OR-09-network_optimization.ipynb)

## 🛠️ Funciones Reutilizables

In [ ]:
def solve_production_plan(
    products: list,
    periods: list,
    demand: dict,
    prod_cost: dict,
    inv_cost: dict,
    machine_hours: dict,
    machine_capacity: float,
    storage_capacity: float
) -> tuple:
    """
    Resuelve problema de production scheduling.
    
    Returns:
        (status, objective_value, production_vars, inventory_vars)
    """
    model = pulp.LpProblem("Production_Plan", pulp.LpMinimize)
    
    # Variables
    X = pulp.LpVariable.dicts("Prod", ((p, t) for p in products for t in periods), lowBound=0)
    I = pulp.LpVariable.dicts("Inv", ((p, t) for p in products for t in periods), lowBound=0)
    
    # Objetivo
    model += pulp.lpSum(
        prod_cost[p] * X[(p, t)] + inv_cost[p] * I[(p, t)]
        for p in products for t in periods
    )
    
    # Restricciones
    for p in products:
        for t in periods:
            if t == periods[0]:
                model += I[(p, t)] == X[(p, t)] - demand[(p, t)]
            else:
                model += I[(p, t)] == I[(p, t-1)] + X[(p, t)] - demand[(p, t)]
    
    for t in periods:
        model += pulp.lpSum(machine_hours[p] * X[(p, t)] for p in products) <= machine_capacity
        model += pulp.lpSum(I[(p, t)] for p in products) <= storage_capacity
    
    # Resolver
    model.solve(pulp.PULP_CBC_CMD(msg=0))
    
    return pulp.LpStatus[model.status], pulp.value(model.objective), X, I

# Ejemplo de uso:
# status, cost, prod, inv = solve_production_plan(products, periods, demand_dict, ...)

## 📝 Notas de Operación (Costes, Retención, Gobernanza)

**Costes**
- Consideraciones de almacenamiento/cómputo/visualización.

**Retención**
- Política por zonas (raw/curated/analytics) y ventanas temporales.

**Gobernanza**
- Calidad de datos, seguridad/PII, linaje, versionado de modelos/artefactos.